<h3 style="color: #3498db;"> 2.1.1 入门 </h3>

In [ ]:
#检测pytorch和cuda版本号
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [ ]:
y1=torch.zeros((2,3,4))#创建一个全零张量
y2=torch.ones((2,3,4))#创建一个全一张量

print(y1.shape)
print(y1.numel())


torch.Size([2, 3, 4])
24


In [ ]:
torch.randn(3,4)#创建一个服从标准正态分布的张量
torch.tensor([[2,1,4,3],[1,2,3,4],[4,3,2,1]])#赋予确定的元素值

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

<h3 style="color: #3498db;"> 2.1.2 运算符 </h3>

* **逐元素运算**：`+` `-` `*` `/` `**` 均为 Shape 严格对齐的逐元素操作，底层由 CUDA 万核并发执行。
* **拼接 (Concatenate)**：`torch.cat((X, Y), dim=d)` 沿第 `d` 维物理拼接，该维度长度相加，其余维度必须严格一致。
* **逻辑判断**：`X == Y` 逐元素比较，返回同 Shape 的布尔张量（`True/False`）。
* **全局求和**：`X.sum()` 将所有元素坍缩为单个标量张量，Shape 从 $(m, n)$ 降维至 $()$。

In [ ]:
x=torch.tensor([1.0,2,4,8])
y=torch.tensor([2,2,2,2])
x+y,x-y,x*y,x/y,x**y

torch.exp(x)

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

In [3]:
X=torch.arange(12,dtype=torch.float32).reshape((3,4))#定义X变量
Y=torch.tensor([[2.0,1,4,3],[1,2,3,4],[4,3,2,1]])#定义Y变量
torch.cat((X,Y),dim=0),torch.cat((X,Y),dim=1)#dim=0-行连结，dim=1-列连结

X==Y#判断是否相等，相等为true，否则为false

X.sum()#求和，变成一个单元素张量

tensor(66.)

<h3 style="color: #3498db;"> 2.1.3 广播机制 (Broadcasting) </h3>

**触发条件**：两个 Shape 不同的张量执行逐元素运算时，底层自动沿**长度为 1 的维度**进行虚拟复制对齐。

* **物理动作**：不实际分配新内存，仅在计算时"假装"沿缺失维度复制扩展，开销为零。
* **Strang 映射**：本质是向量空间中的外积式升维——列向量沿行方向广播 = 将 $\mathbb{R}^{m \times 1}$ 虚拟扩展为 $\mathbb{R}^{m \times n}$。
* **工程陷阱**：当两个张量的维度既不相等也不为 1 时，广播失败，直接抛出 Shape 不匹配的 RuntimeError。

In [ ]:
a=torch.arange(3).reshape((3,1))
b=torch.arange(2).reshape((1,2))
print(a,'\n',b)
a+b# a->3*2 ,b->3*2

tensor([[0],
        [1],
        [2]]) 
 tensor([[0, 1]])


tensor([[0, 1],
        [1, 2],
        [2, 3]])

<h3 style="color: #3498db;"> 2.1.4 索引和切片 </h3>

* **单点索引** `x[i, j]`：触发**维度坍缩**，被索引的维度直接消失，返回降维后的子张量。
* **切片索引** `x[i:j]`：**强制保留维度骨架**，即使只切出一个元素，Shape 中该维度仍以长度 1 存在。
* **左闭右开铁律**：`x[a:b]` 物理上提取索引 $[a, b)$，即包含 $a$ 但不包含 $b$。
* **赋值广播**：`x[0:2, :] = 12` 底层对右侧标量触发广播，批量覆写指定区域的所有元素。

In [5]:
X[-1],X[1:3]#[-1]选择最后一行元素,[-3]选择第二行和第三行元素
X[1,2]=9#单元素赋值
X

X[0:2,:]=12
X

tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])

<h3 style="color: #3498db;"> 2.1.5 张量的内存管理 (In-place Operations) </h3>

**核心痛点**：非原地操作频繁开辟新显存，极易引发碎片化与 OOM 崩溃。

| 操作 | 底层机制 | 指针变化 | 显存开销 |
|------|----------|----------|----------|
| `Y = Y + X` | 新开辟完整内存，指针改向 | ✅ 改变 | 最差 |
| `Y[:] = Y + X` | 临时内存计算后拷贝回原地址 | ❌ 不变 | 峰值有开销 |
| `Y += X` / `Y.add_(X)` | 直接在原内存格子内覆写，零临时分配 | ❌ 不变 | **零额外开销，工程最优解** |

In [ ]:
before=id(Y)
Y=Y+X#非原地操作，重新分配内存
id(Y)==before#返回false

Y+=X#原地操作，内存地址不变
Y[:]=Y+X#同上
id(Y)==before

True

In [ ]:
#示例：
Z=torch.zeros_like(Y)
print('id(Z):',id(Z))
Z[:]=X+Y
print('id(Z):',id(Z))

id(Z): 2164800333328
id(Z): 2164800333328


<h3 style="color: #3498db;"> 2.1.6 转换为其他 Python 对象 </h3>

### 核心结论：三条 API 边界

| 场景 | API | 输出类型 | 底层行为 |
|------|-----|----------|----------|
| Tensor → NumPy | `.numpy()` | `np.ndarray` | **共享底层内存**，原地修改互相影响 |
| NumPy → Tensor | `torch.tensor(A)` | `torch.Tensor` | 同上，共享内存 |
| 单元素张量 → 纯数字 | `.item()` | Python `float/int` | 斩断计算图，安全提取（防 OOM）|
| 多元素张量 → 纯列表 | `.tolist()` | Python `list` | 批量剥离为原生嵌套列表 |

In [ ]:
#type()验证格式转换
A=X.numpy()
B=torch.tensor(A)
type(A),type(B)

(numpy.ndarray, torch.Tensor)

In [ ]:
#转换大小为1的张量
a=torch.tensor([3.5])
a,a.item(),float(a),int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

In [ ]:
x=torch.tensor([[1,2],[3,4]])
x[1,1].item()#提取单元素
x.tolist()#提取列表

[[1, 2], [3, 4]]

<h1 align="center"> 🛠️ 2.2 数据预处理 </h1>
<hr>

<h3 style="color: #3498db;"> 2.2.1 读取数据集 </h3>

### 1. 文件系统 I/O (`os` 模块)
* `os.path.join`：跨平台路径拼接，自动适配 `\` 或 `/`。`..` = 返回上一级父目录。
* `os.makedirs`：附带 `exist_ok=True`，目录已存在时静默跳过，不抛异常。
* `with open(...) as f`：上下文管理器，无论成功或报错，自动关闭文件句柄，杜绝资源泄漏。

### 2. Pandas 读取 (`pd.read_csv`)
* **工程定位**：Pandas 专司数据清洗脏活，清洗完毕后必须转化为 PyTorch Tensor 才能送入模型。
* **读取机制**：严格服从变量中的物理路径，自动解析首行为表头列名，自动将空缺内容映射为 `NaN`。

In [6]:
#写入数据集
import os

os.makedirs(os.path.join('..','data'),exist_ok=True)
data_file=os.path.join('..','data','house_tiny.csv')
with open(data_file,'w') as f:
    f.write('NumRooms,Alley,Price\n')#列名
    f.write('NA,Pave,127500\n')#每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')


In [7]:
#!pip install pandas #安装pandas
import pandas as pd
#读取数据集
data=pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


<h3 style="color: #3498db;"> 2.2.2 处理缺失值 (Data Preprocessing) </h3>
神经网络底层拒收缺失值（`NaN`）与文本。本节核心是将残缺业务表格淬炼为纯净张量：

### 1. 架构解耦 ($X$ 与 $Y$)
* **物理动作**：通过 `.iloc` 坐标切片，将数据强行切断为“输入特征 $X$” (`inputs`) 与“预测目标 $Y$” (`outputs`)。
* **工程铁律**：后续所有预处理只能针对 $X$。绝不允许 $Y$ 混入，严防数据泄露（Data Leakage）。

### 2. 均值插补 (数值特征)
* **物理动作**：调用 `.fillna()`，用数值列的**均值**覆盖 `NaN` 空洞，维持分布的“几何质心”。
* **断层规避**：Pandas 2.0+ 必须显式下达 `numeric_only=True` 指令，强制跳过非数值列。

### 3. 独热编码 (文本特征)
* **物理动作**：调用 `pd.get_dummies()`，将 $N$ 种文本状态撕裂为 $N$ 维正交基（独立的 0/1 特征列）。
* **情报固化**：开启 `dummy_na=True`，将 `NaN` 视为独立状态保留。在工业界，数据的缺失行为本身即是强特征。

In [8]:
#神经网络架构Y=WX
# inputs 接管了第 0 列和第 1 列（房间数和巷子类型)--X
# outputs 独占了第 2 列（价格）--Y
inputs,outputs=data.iloc[:,0:2],data.iloc[:,2]
print(inputs.dtypes)
inputs=inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)

NumRooms    float64
Alley        object
dtype: object
   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


In [9]:
#独热编码（One-Hot Encoding）
inputs=pd.get_dummies(inputs,dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


<h3 style="color: #3498db;"> 2.2.3 转换为张量格式 (核心数据流) </h3>
数据在送入 GPU 前，必须经历三次**严格单向**的物理蜕变：

1. **Pandas DataFrame**（初始 `inputs`）：**业务清洗台**。保留列名，包容 `NaN` 和文本，专职负责特征解耦与清洗。
2. **NumPy Array**（`.to_numpy()` 算子）：**剥离业务属性**。强行扒掉列名与行号，退化为绝对纯粹的浮点数连续内存块。
3. **PyTorch Tensor**（`torch.tensor()`）：**GPU 算力弹药**。赋予底层求导（Gradient）器官，具备送入显卡执行万核并发矩阵乘法（$Y=WX$）的物理资格。

> **工程铁律**：Pandas 只干清洗脏活，Tensor 专责核心算力。物理边界森严，绝不混用。

In [10]:
import torch
#注意：需先进行上文的独热编码，把文本撕裂成 True/False 的数字维度
X=torch.tensor(inputs.to_numpy(dtype=float))
y=torch.tensor(outputs.to_numpy(dtype=float))
X,y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

<h1 align="center"> 📐 2.3 线性代数 (Linear Algebra) </h1>
<hr>

<h3 style="color: #3498db;"> 2.3.1 标量——由单个元素表示 </h3>

In [11]:
#单元素标量运算
import torch
x=torch.tensor(3.0)
y=torch.tensor(2.0)
x+y,x*y,x/y,x**y

(tensor(5.), tensor(6.), tensor(1.5000), tensor(9.))

<h3 style="color: #3498db;"> 2.3.2 向量——一维张量 </h3>

In [ ]:
#创建一维张量
x=torch.arange(4)
x

tensor([0, 1, 2, 3])